# Assignment 1: A small language pipeline

You will turn a book PDF into a corpus, tokenise it, learn byte-pair encoding (BPE) and train n-gram language models. BPE here is the classic character-level algorithm, not byte-level BPE.

Write your code in `src/` and in the notebook cells marked `TODO`. Submit the notebook with all outputs.

Use only the Python standard library, NumPy, pandas, Matplotlib and `pypdf`. Do not use NLP packages such as NLTK or spaCy, or any tokeniser, BPE or language-model library.

| Task | Marks |
|---|---:|
| 1. Corpus | 10 |
| 2. Tokenisation | 10 |
| 3. Data split | 6 |
| 4. Zipf plot and type-token ratio | 6 |
| 5. Learn BPE | 14 |
| 6. Apply BPE | 8 |
| 7. n-gram models | 18 |
| 8. Generate and evaluate | 10 |
| 9. Hand-worked calculations | 12 |
| 10. Short questions | 6 |
| **Total** | **100** |

## Setup
Run this cell at the start of every session. In Colab it mounts Google Drive, moves to your copy of `assignment1` and installs the requirements. Change the path if you cloned elsewhere. Locally it only reloads `src/` automatically after you edit it. See the README for first-time setup.

In [ ]:
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/cng463-assignments/assignment1
%pip install -q -r requirements.txt
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import Counter
import math
import random
import pandas as pd
import matplotlib.pyplot as plt
from src.text import (extract_pages, clean_pages, normalise_text, split_into_sentences, tokenise,
                        save_sentences, load_sentences, deduplicate, split_blocks, corpus_stats)
from src.bpe import learn_bpe, encode_word
from src.lm import NGramLM, BOS, EOS, UNK

## 1. Corpus (10 marks)
Choose a book in PDF form with a text layer. Scanned pages will not work. Public-domain or openly licensed books are the safest choice, for example from Project Gutenberg, Wikisource or the Internet Archive.

Choose a page range with 5,000–50,000 words and at least 300 sentences excluding front matter such as the contents and preface. Put the PDF in `data/`.

Implement `clean_pages`, `normalise_text` and `split_into_sentences` in `src/text.py`. Cleaning should remove page numbers, running headers and headings, rejoin words split by a hyphen at a line end, and replace ligatures (single characters that combine multiple letters, such as ﬁ) with separate letters (e.g. ﬁ → fi).

In [ ]:
# TODO [4 points]: Set your PDF and page range, describe the corpus and compare a raw and a cleaned page.
PDF_PATH, FIRST_PAGE, LAST_PAGE = "data/your_book.pdf", 1, 1
raw_pages = extract_pages(PDF_PATH, FIRST_PAGE, LAST_PAGE)
cleaned_pages = clean_pages(raw_pages)

In [ ]:
# TODO [6 points]: Segment and save the sentences, then inspect them.
sentence_texts = split_into_sentences(normalise_text(clean_pages(raw_pages)))
save_sentences(sentence_texts, "data/corpus.txt")

## 2. Tokenisation (10 marks)
Implement `tokenise` in `src/text.py` with regular expressions (6). Show at least four edge cases, preferably from your book (2). Explain how you handle case, punctuation, apostrophes and numbers, and give one limitation for your corpus language (2).

In [ ]:
# TODO [Task 2]: Show edge cases and explain your choices.

## 3. Data split (6 marks)
Nearby sentences in a book share names and topics, so a random sentence split leaks information into the test set. Implement `deduplicate` and `split_blocks` in `src/text.py`.
- Remove repeated tokenised sentences, report how many you removed and show that the three sets share no sentence (2).
- Shuffle blocks of 10 consecutive sentences with seed 42 and split the blocks 80/10/10 (2).
- State what is fitted on training data only and one leakage risk that remains (2).

In [ ]:
# TODO [Task 3]: Remove duplicates, split by blocks and check for overlap.
sentences = load_sentences("data/corpus.txt")

## 4. Zipf plot and type-token ratio (6 marks)
Implement `corpus_stats` and report N, V and TTR = V/N for the training set. Count every token, including punctuation (2). Plot frequency against rank with log axes and axis labels (2). Explain the shape of the plot and why TTR depends on corpus size (2).

In [ ]:
# TODO [Task 4]: Compute training statistics and plot rank against frequency.

## 5. Learn BPE (14 marks)
Implement `merge_symbols` and `learn_bpe` in `src/bpe.py`. Start each training word type as its characters plus `</w>`.
- Count adjacent pairs inside each word, weighted by the word's training frequency (5).
- Pick the most frequent pair. Break ties by the smaller `(left, right)` tuple in Python's ordering, so `'</w>'` comes before letters. Merge left to right without overlaps (5).
- Learn 200 merges, show the first 10 with their counts and add one check that tests ties or overlaps (4).

In [ ]:
# TODO [Task 5]: Learn 200 merges, show the first 10 and test your code.

## 6. Apply BPE (8 marks)
Implement `encode_word` (3). Encode five training words and five test words, including words not seen in training. Report the mean number of pieces per test token, counting `</w>` (3). Check that the pieces rebuild every test token and explain what happens to a character never seen in training (2).

In [ ]:
# TODO [Task 6]: Encode words, measure pieces per token and check reconstruction.

## 7. n-gram models (18 marks)
Implement `NGramLM` in `src/lm.py` for n = 1, 2 and 3.
- Build the vocabulary from training data only. Keep tokens seen at least twice and map the rest to `<unk>`. Add `<unk>` and `</s>`. `<s>` is used only as context (4).
- Pad each sentence with n−1 `<s>` and end it with one `</s>`. Count each context and its next token, never across sentences (6).
- Use P(w | h) = (C(h, w) + α) / (C(h) + α|V|), with the same V for all orders (5).
- Check that the probabilities sum to one for a seen and an unseen context (3).

In [ ]:
# TODO [Task 7]: Fit the three models and check that each distribution sums to one.

## 8. Generate and evaluate (10 marks)
Implement `perplexity` and `generate` (5). Perplexity is exp(−mean log probability) over all predicted tokens, including one `</s>` per sentence but not `<s>`. Generation samples from the full smoothed distribution and stops at `</s>` or after 40 tokens.

For each order, choose α from {0.001, 0.01, 0.1, 1} by development perplexity. Report the test perplexity, the test unknown rate, the number of test sentences and tokens, and two samples with seeds 7 and 21 (5).

In [ ]:
# TODO [Task 8]: Choose alpha on dev, evaluate once on test and generate.

## 9. Hand-worked calculations (12 marks)
Work these out by hand first. Type your working in the next cell or insert a clear photo. Then run the check cell.

**BPE (6).** The word counts are `pat` 4, `tap` 3, `taps` 2 and `spat` 5. Use the conventions of Task 5. Give all initial pair counts, then the first three merges and the symbols of each word after each merge.

**Bigram (6).** Train on the sentences `b a` and `a b b` with min_count = 1 and α = 1. Give the vocabulary and the counts you need. Calculate P(b | <s>), P(a | b), P(</s> | a) and P(c | b), where c is a new word. Then give the probability and perplexity of the sentence `b a`, including its `</s>`.

*Your working here.*

In [ ]:
# Check your hand-worked answers with your code.
print(learn_bpe(Counter({"pat": 4, "tap": 3, "taps": 2, "spat": 5}), 3))
toy = NGramLM(2, alpha=1, min_count=1).fit([["b", "a"], ["a", "b", "b"]])
print([toy.probability("b", (BOS,)), toy.probability("a", ("b",)), toy.probability(EOS, ("a",)), toy.probability("c", ("b",))])
print(toy.perplexity([["b", "a"]]))

## 10. Short questions (6 marks)
Answer each question with a word, a number or a short sequence of symbols. Each answer earns one mark.
1. Apply the single merge (a, a) to the word `aaa`. Which symbols do you get?
2. A test word contains a character that never appears in training. How many BPE pieces contain that character?
3. Over how many prediction events does your trigram test perplexity average?
4. As α grows very large, what value does every model's perplexity approach? Give the number for your models.
5. Is the TTR of the first 1,000 training tokens higher than, lower than or equal to the TTR of the whole training set?
6. What is the probability that your selected unigram model generates an empty sentence? Round to three decimals.

In [ ]:
# TODO [Task 10]: Replace each None with your answer.
ANSWERS = {1: None, 2: None, 3: None, 4: None, 5: None, 6: None}